<a href="https://colab.research.google.com/github/leejuny0ng/AI/blob/main/economy_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [10]:
from logging import Formatter
import urllib.request
import urllib.parse
import json
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from datetime import datetime
import numpy as np
from collections import Counter
import networkx as nx

def count_query_frequency(text, query):
    """쿼리 단어의 빈도수를 계산"""
    return len(re.findall(query.lower(), text.lower()))

def create_sentence_similarity_matrix(sentences):
    """문장 간 유사도 행렬 생성"""
    n = len(sentences)
    similarity_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i != j:
                similarity_matrix[i][j] = calculate_sentence_similarity(sentences[i], sentences[j])

    return similarity_matrix

def calculate_sentence_similarity(sent1, sent2):
    """두 문장 간의 유사도 계산"""
    words1 = set(word_tokenize(sent1.lower()))
    words2 = set(word_tokenize(sent2.lower()))

    # 불용어 제거
    stop_words = set(stopwords.words('english'))
    words1 = words1.difference(stop_words)
    words2 = words2.difference(stop_words)

    # Jaccard 유사도 계산
    intersection = len(words1.intersection(words2))
    union = len(words1.union(words2))

    return intersection / union if union != 0 else 0

def summarize_text(text, num_sentences=3):
    """PageRank 알고리즘을 사용한 텍스트 요약"""
    # 문장 분리
    sentences = sent_tokenize(text)
    if len(sentences) <= num_sentences:
      # 만약 문장 수가 요약할 문장 수보다 적거나 같으면 각 문장을 순서대로 처리
      formatted_text = ''
      for sentence in sentences:
        sentence = sentence.strip()
        if sentence.endswith('.'):
          formatted_text += sentence + '\n'
        else:
          formatted_text += sentence + ' '
      return formatted_text.strip()

    # 유사도 행렬 생성
    similarity_matrix = create_sentence_similarity_matrix(sentences)

    # PageRank 적용
    nx_graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(nx_graph)

    # 상위 문장 선택
    ranked_sentences = [(scores[i], s) for i, s in enumerate(sentences)]
    ranked_sentences.sort(reverse=True)

    # 원본 순서 유지를 위해 인덱스 기반으로 정렬
    selected_sentences = []
    for _, sentence in ranked_sentences[:num_sentences]:
        original_idx = sentences.index(sentence)
        selected_sentences.append((original_idx, sentence))
    selected_sentences.sort()


    # 선택된 문장들을 마침표 기준으로 개행하여 반환
    formatted_text = ''
    for _, sentence in selected_sentences:
        sentence = process_sentence(sentence)
        if sentence.endswith('.'):
            formatted_text += sentence + '\n'
        else:
            formatted_text += sentence + ' '

    return formatted_text.strip()

    # return '\n'.join(sentence for _, sentence in selected_sentences)

def get_naver_news(query, client_id, client_secret):
    """
    Fetch news from Naver API with proper authentication and enhanced processing
    """
    enc_text = urllib.parse.quote(query)
    url = f"https://openapi.naver.com/v1/search/news.json?query={enc_text}&display=100"

    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)

    try:
        response = urllib.request.urlopen(request)

        if response.getcode() == 200:
            response_body = response.read().decode('utf-8')
            news_items = json.loads(response_body)['items']

            print(f'\n{datetime.now().date()} / 당신이 고른 단어({query})를 토대로 검색한 오늘의 기사\n')
            economy_news = [
                item for item in news_items if is_today(item['pubDate']) and ("sid=101" in item['link'])
            ]

            # 기사 정보와 query 빈도수를 함께 저장
            news_with_frequency = []
            for news in economy_news:
                raw_title = remove_html_tags(news['title']).strip()
                raw_description = remove_html_tags(news['description']).strip()

                # query 빈도수 계산 (제목과 내용에서)
                total_frequency = (
                    count_query_frequency(raw_title, query) * 2 +  # 제목의 빈도수는 2배 가중치
                    count_query_frequency(raw_description, query)
                )

                # 내용 요약
                summarized_text = summarize_text(raw_description)

                news_with_frequency.append({
                    'frequency': total_frequency,
                    'title': raw_title,
                    'link': news['link'],
                    'description': summarized_text
                })

            if len(news_with_frequency) == 0:
                return f"해당 단어({query})를 포함한 경제 뉴스가 없습니다.\n"

            # 빈도수 기준으로 정렬
            news_with_frequency.sort(key=lambda x: x['frequency'], reverse=True)

            # 포맷팅
            formatted_news = []
            for idx, news in enumerate(news_with_frequency, 1):
                formatted_news.append(
                    f"[중요도 순위: {idx}] (키워드 빈도수: {news['frequency']})\n"
                    f"Title: {news['title']}\n"
                    f"Link: {news['link']}\n"
                    f"Summary: {news['description']}\n"
                )

            return "\n".join(formatted_news)

        else:
            return f"Error Code: {response.getcode()}"

    except urllib.error.HTTPError as e:
        return f"HTTP Error: {e.code}\nReason: {e.reason}"
    except Exception as e:
        return f"Error: {str(e)}"

def is_today(pub_date):
    today = datetime.now().date()
    pub_date_obj = datetime.strptime(pub_date, '%a, %d %b %Y %H:%M:%S %z').date()
    return pub_date_obj == today

def remove_html_tags(text):
    text = re.sub(r'<br[^>]*>', '\n', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def process_sentence(sentence):
    """
    문장을 정리하고 마침표 위치를 확인하는 함수
    """
    sentence = sentence.strip()
    # 문장 중간의 마침표를 처리 (예: "주식회사.)
    common_abbreviations = [
        r'[A-Z][A-Z]\.', # CO.
        r'[A-Z]\.[A-Z]',  # U.S, U.K,
        r'[A-Z]\.[A-Z]\.[A-Z]',  # F.B.I
        r'[A-Z][a-z]+\.',  # Mr., Dr.
        r'[A-Z]+',  # NASA
        r'[A-Za-z]+\.[A-Za-z]+',  # Ph.D
        r'[A-Za-z]+\.[A-Za-z]+\.[A-Za-z]+'  # D.N.A
    ]

    # 약어 패턴을 임시 토큰으로 치환
    replacements = {}
    for i, pattern in enumerate(common_abbreviations):
        matches = re.finditer(pattern, sentence)
        for match in matches:
            abbr = match.group()
            temp_token = f"@ABBR{i}{len(replacements)}@"
            replacements[temp_token] = abbr
            sentence = sentence[:match.start()] + temp_token + sentence[match.end():]

    # 이제 남은 마침표는 문장 구분용이므로 공백 추가
    sentence = re.sub(r'\.(?=[^ ])', '. ', sentence)

    # 임시 토큰을 다시 원래 약어로 복원
    for temp_token, original in replacements.items():
        sentence = sentence.replace(temp_token, original)

    return sentence

def get_single_word_query():
    while True:
        query = input("검색할 키워드를 한 단어로 입력하세요(quit 을 입력하면 종료합니다.): ").strip()
        if len(query.split()) == 1:
            return query
        else:
            print("\n한 단어만 입력해 주세요. 다시 시도하세요.")

# 메인 실행 코드
if __name__ == "__main__":
    while True:
        query = get_single_word_query()
        if query == 'quit':
            print('종료합니다.')
            break
        client_id = "XyeQvys3L4YHfDlDXQu3"
        client_secret = "Igk0kU1ThL"
        result = get_naver_news(query, client_id, client_secret)
        print(result)

KeyboardInterrupt: Interrupted by user